# RAG Document Assistant: Querying the EU AI Act with LangChain

This project builds a conversational RAG (Retrieval-Augmented Generation) agent using LangChain and a Groq-hosted LLM to enable question-answering over the EU AI Act. 

Rather than relying on the model's training data alone, the agent retrieves relevant passages directly from the document to ground its answers. This assistant helps users navigate a large legislative document quickly and accurately, without needing to upload the file externally.

----


## Setup

Dependencies are listed in `requirements.txt`. Install with:

    pip install -r requirements.txt

Libraries used in this project:

- [`langchain`](https://www.langchain.com/) for chain orchestration, retrieval, and prompt management
- [`langchain-groq`](https://github.com/langchain-ai/langchain/tree/master/libs/partners/groq) for integrating Groq-hosted LLMs via the Groq Inference API
- [`langchain-huggingface`](https://github.com/langchain-ai/langchain-huggingface) for HuggingFace embedding models
- [`langchain-community`](https://github.com/langchain-ai/langchain-community) for community-maintained loaders and vector store integrations
- [`sentence-transformers`](https://www.sbert.net/) for encoding text chunks into dense vector embeddings
- [`chromadb`](https://www.trychroma.com/) for storing and retrieving document embeddings as a local vector store
- [`python-dotenv`](https://pypi.org/project/python-dotenv/) for loading API keys securely from a `.env` file

In [1]:
import os
import warnings
warnings.filterwarnings('ignore')

from dotenv import load_dotenv
load_dotenv()  # loads API keys from .env file

from langchain_community.document_loaders import PDFPlumberLoader
from langchain_text_splitters import CharacterTextSplitter
from langchain_community.vectorstores import Chroma
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from langchain_core.messages import HumanMessage, AIMessage
from langchain.chains import create_history_aware_retriever, create_retrieval_chain
from langchain.chains.combine_documents import create_stuff_documents_chain

## Document Preprocessing & Indexing

Loading the EU AI Act; Chunking; Preview; Embedding.

`PDFPlumberLoader` is used instead of the default `PyPDFLoader`, since EU legislative PDFs use complex font encoding that causes character-level extraction errors with the latter.

In [2]:
loader = PDFPlumberLoader("data/eu_ai_act.pdf")
documents = loader.load()
print(f"Loaded {len(documents)} pages")

Loaded 144 pages


In [3]:
text_splitter = CharacterTextSplitter(
    chunk_size=1500,
    chunk_overlap=200,
    separator="\n"
)
texts = text_splitter.split_documents(documents)
print(len(texts))

502


In [4]:
# Preview document
print(f"Total pages loaded: {len(documents)}")
print(f"Total chunks after splitting: {len(texts)}")
print("\n--- Sample chunk ---")
print(texts[5].page_content)

Total pages loaded: 144
Total chunks after splitting: 502

--- Sample chunk ---
ultimate aim of increasing human well-being.
(7) In order to ensure a consistent and high level of protection of public interests as regards health, safety and
fundamental rights, common rules for high-risk AI systems should be established. Those rules should be consistent
with the Charter, non-discriminatory and in line with the Union’s international trade commitments. They should
also take into account the European Declaration on Digital Rights and Principles for the Digital Decade and the
Ethics guidelines for trustworthy AI of the High-Level Expert Group on Artificial Intelligence (AI HLEG).
(8) A Union legal framework laying down harmonised rules on AI is therefore needed to foster the development, use
and uptake of AI in the internal market that at the same time meets a high level of protection of public interests, such
as health and safety and the protection of fundamental rights, including democracy

#### Embedding and storing

The following code creates a local embedding model from HuggingFace using `sentence-transformers` and ingests the document chunks into a Chroma vector store.

In [5]:
embeddings = HuggingFaceEmbeddings()  # uses all-MiniLM-L6-v2 by default
docsearch = Chroma.from_documents(texts, embeddings)
print('document ingested')

document ingested


## LLM Construction
We use Groq's inference API to access **Llama 3.3 70B** via `ChatGroq`. Groq provides fast, free-tier access to open-source models without requiring local GPU resources. The API key is loaded securely from the `.env` file. 

In [8]:
llm = ChatGroq(
    model="llama-3.3-70b-versatile", 
    temperature=0.1,
    max_tokens=512,
    api_key=os.getenv("GROQ_API_KEY")
)

## Retrieval with LangChain

Here is a simple Q&A application over the document source using LangChain's LCEL (LangChain Expression Language). The retriever fetches relevant chunks from the vector store, which are passed as context to the LLM alongside the user's question.

In [12]:
from langchain_core.runnables import RunnablePassthrough, RunnableLambda

retriever = docsearch.as_retriever()

def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

qa_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | RunnableLambda(lambda x: f"Context:\n{x['context']}\n\nQuestion: {x['question']}")
    | llm
    | StrOutputParser()
)

query = "How does the EU AI Act define AI systems?"
result = qa_chain.invoke(query)
print(result)

The provided text does not explicitly define AI systems. However, it mentions various aspects of the EU AI Act, such as requirements for AI systems, implementation, and amendments to existing regulations. 

To find the definition of AI systems, you would need to refer to the main text of the EU AI Act, specifically Article 2 or other relevant sections that provide definitions. The EU AI Act likely includes a definition of AI systems, but it is not provided in the given context. 

If you're looking for the definition, I recommend checking the official EU AI Act document or a reliable source that provides information on the Act.


In [13]:
retriever = docsearch.as_retriever()

prompt = ChatPromptTemplate.from_template("""
Answer the question based only on the following context.
If the answer is not in the context, say you don't know.

Context:
{context}

Question: {question}
""")

def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

qa_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

query = "How does the EU AI Act define AI systems?"
result = qa_chain.invoke(query)
print(result)

I don't know. The provided context does not contain a definition of AI systems.


The cell below shows that without conversation memory, the model cannot answer a follow-up question that refers to the previous exchange. The pronoun "it" has no context to resolve against.

In [14]:
query = "What are the prohibited uses in it?"
result = qa_chain.invoke(query)
print(result)

I don't know. The context mentions the prohibition of AI practices referred to in Article 5, but it does not specify what those prohibited uses are.


## Conversational Retrieval with Memory

To give the model memory of previous exchanges, we use two components:

1. **`create_history_aware_retriever`**: rewrites the user's query into a standalone question using the chat history, so the retriever can find the right documents even when the question contains pronouns or references to prior turns.
2. **`create_retrieval_chain`**: combines the history-aware retriever with an answer-generation chain that also has access to the chat history.

In [16]:
# Prompt for answering questions with retrieved context and chat history
qa_prompt = ChatPromptTemplate.from_messages([
    ("system",
     "You are an assistant for question-answering tasks. "
     "Use the following retrieved context to answer the question. "
     "If you don't know the answer, say so. Keep answers concise.\n\n"
     "{context}"),
    MessagesPlaceholder("chat_history"),
    ("human", "{input}"),
])

question_answer_chain = create_stuff_documents_chain(llm, qa_prompt)
conversational_rag_chain = create_retrieval_chain(
    history_aware_retriever, question_answer_chain
)

Chat history is maintained as a list of `HumanMessage` and `AIMessage` objects and updated after each turn.

In [17]:
chat_history = []

query = "How does the EU AI Act define AI systems?"
result = conversational_rag_chain.invoke({"input": query, "chat_history": chat_history})
print(result["answer"])

chat_history.extend([HumanMessage(content=query), AIMessage(content=result["answer"])])

The provided context does not explicitly define AI systems. It discusses various aspects of the EU AI Act, such as implementation, requirements, and amendments to existing regulations, but does not provide a definition of AI systems.


In [18]:
# Follow-up question — the model resolves "it" using the chat history
query = "What are the prohibited uses in it?"
result = conversational_rag_chain.invoke({"input": query, "chat_history": chat_history})
print(result["answer"])

chat_history.extend([HumanMessage(content=query), AIMessage(content=result["answer"])])

The provided context does not explicitly list the prohibited uses of AI systems in the EU AI Act. However, it mentions that the Act aims to "prohibit certain unacceptable AI practices" (paragraph 26), but it does not specify what those practices are.


In [19]:
query = "What is the aim of it?"
result = conversational_rag_chain.invoke({"input": query, "chat_history": chat_history})
print(result["answer"])

chat_history.extend([HumanMessage(content=query), AIMessage(content=result["answer"])])

The aim of the EU AI Act is to promote the uptake of human-centric and trustworthy artificial intelligence, while ensuring a high level of protection of health, safety, fundamental rights, democracy, the rule of law, and environmental protection, and to support innovation. The ultimate aim is to increase human well-being.


### Wrap up and make it an agent

The following code defines a function that wraps the conversational RAG chain in an interactive loop. The agent maintains chat history across turns and can retrieve information from the document to answer follow-up questions. Type `quit`, `exit`, or `bye` to stop.

In [20]:
def qa_agent():
    chat_history = []
    while True:
        query = input("Question: ")
        if query.lower() in ["quit", "exit", "bye"]:
            print("Answer: Goodbye!")
            break
        result = conversational_rag_chain.invoke({
            "input": query,
            "chat_history": chat_history
        })
        answer = result["answer"]
        chat_history.extend([HumanMessage(content=query), AIMessage(content=answer)])
        print("Answer:", answer)

By running the cell below, you can ask the agent any question regarding the EU AI Act.
To **stop** the agent, type `quit`, `exit`, or `bye`.

In [21]:
qa_agent()

Answer: The EU AI Act aims to prevent the harmful usage of AI by establishing common rules for high-risk AI systems, ensuring they meet a high level of protection for health, safety, and fundamental rights. This includes:

1. Laying down harmonized rules on AI to protect public interests.
2. Regulating the placing on the market, putting into service, and use of certain AI systems.
3. Establishing requirements for high-risk AI systems to ensure they are safe and trustworthy.
4. Providing a safety net for non-high-risk AI systems through other regulations, such as Regulation (EU) 2023/988.
5. Ensuring enforcement through penalties and other measures for non-compliance.

These measures aim to promote the development and use of trustworthy AI while protecting against harmful effects.
Answer: The EU AI Act addresses bias in AI by requiring providers of high-risk AI systems to:

1. Implement measures to detect and correct bias in their systems.
2. Process special categories of personal data 